# Environment 3: Taxi-v3

So sánh Q-Learning và Expected SARSA theo cùng **environment-step budget**. Trọng tâm là sample efficiency, chất lượng greedy policy và illegal pickup/dropoff rate.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/Briannguyen3106/project1.git'
ROOT = Path('/kaggle/working/project1')

if ROOT.exists():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(ROOT)], check=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.Environment3 as experiment
experiment.OUTPUT_DIR = Path('/kaggle/working/results/figures/environment3_taxi')
run_experiment = experiment.run_experiment
OUTPUT_DIR = experiment.OUTPUT_DIR
print(f'Project root: {ROOT}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
%cd /kaggle/working/project1
!pip install -r requirements.txt

## Chạy thí nghiệm chính

Mỗi thuật toán chạy 30 seed và đúng 500.000 training transitions/seed. Epsilon giảm theo cumulative steps. Greedy evaluation dùng 200 episode với cùng seed panel tại mọi checkpoint.

In [ ]:
episodes_df, evaluations_df, seed_metrics, q_tables = run_experiment(
    seeds=list(range(30)),
    step_budget=500_000,
)
print(f'Results saved to: {OUTPUT_DIR}')

## Sanity check và kết quả cuối

Bảng đầu xác nhận mọi run có cùng interaction budget. Evaluation table báo mean, standard deviation và median qua training seed.

In [ ]:
step_check = (
    episodes_df.groupby(['algorithm', 'seed'])['cumulative_steps']
    .max()
    .groupby('algorithm')
    .agg(['min', 'max'])
)
display(step_check)

final_evaluations = (
    evaluations_df.sort_values(['algorithm', 'seed', 'cumulative_steps'])
    .groupby(['algorithm', 'seed'], as_index=False)
    .tail(1)
)
display(
    final_evaluations.groupby('algorithm')[[
        'eval_return',
        'eval_success_rate',
        'eval_episode_length',
        'eval_illegal_actions',
    ]].agg(['mean', 'std', 'median'])
)

## Sample efficiency

Threshold được định trước là evaluation return >= 5 và phải duy trì trong 3 checkpoint liên tiếp. First success không được dùng làm metric chính vì hai thuật toán có thể giống nhau trước khi reward thành công đầu tiên lan truyền.

In [ ]:
display(
    seed_metrics.groupby('algorithm').agg(
        threshold_reach_rate=('reached_threshold', 'mean'),
        steps_to_threshold_mean=('steps_to_threshold', 'mean'),
        steps_to_threshold_median=('steps_to_threshold', 'median'),
        return_auc_mean=('normalized_return_auc', 'mean'),
        return_auc_std=('normalized_return_auc', 'std'),
    )
)

first_success = (
    episodes_df.groupby(['algorithm', 'seed'], as_index=False)['first_success_step']
    .max()
)
display(first_success.groupby('algorithm')['first_success_step'].agg(['mean', 'std', 'median']))

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/environment3_taxi', 'zip', OUTPUT_DIR)
print(f'Archive ready: {archive}')